### RAG with MongoDB => Data Ingestion , Retrieval and genration pipeline

### Data Ingestion

In [1]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set")

print("GROQ_API_KEY loaded")

GROQ_API_KEY loaded


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

c:\Desktop\RAG-Krish Nayak\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


def get_embedding(text , input_type = 'documents'):
    if input_type == 'query':
        return model.embed_query(text)
    
    return model.embed_documents([text])[0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2354.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
embed = get_embedding("RAG Technology")
print(len(embed))

384


In [5]:
# Data Ingestion

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF

loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()


# Split the data into chunks 

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 20)
documents = text_splitter.split_documents(data)




In [6]:
# Prepare documents for insertion

docs_to_insert = [
    {
        "text" : doc.page_content,
        "embedding" : get_embedding(doc.page_content)
    }for doc in documents]


In [7]:
docs_to_insert

[{'text': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue',
  'embedding': [-0.010066003538668156,
   0.009520412422716618,
   -0.008275438100099564,
   0.03640144690871239,
   -0.031777556985616684,
   -0.04345524311065674,
   -0.10090308636426926,
   0.025192659348249435,
   0.011559919454157352,
   0.059784598648548126,
   -0.06596753001213074,
   0.05412464216351509,
   -0.011226306669414043,
   -0.019266808405518532,
   -0.06192454323172569,
   0.016927504912018776,
   0.020162776112556458,
   -0.10663889348506927,
   0.06556180864572525,
   -0.0021224704105407,
   -0.005065780133008957,
   -0.015644168481230736,
   0.032565515488386154,
   0.007597240153700113,
   0.05290800705552101,
   -0.04959668219089508,
   -0.

In [8]:
from pymongo import MongoClient

# Connect your MongoDB deployment via environment variable
mongo_uri = os.getenv("MONGODB_URI")
if not mongo_uri:
    raise ValueError("MONGODB_URI is not set")

client = MongoClient(mongo_uri)
collection = client["RAG"]["ragpdf"]

# Insert documents into the collection
result = collection.insert_many(docs_to_insert)
result

InsertManyResult([ObjectId('69e23677c033aea568bcbf7b'), ObjectId('69e23677c033aea568bcbf7c'), ObjectId('69e23677c033aea568bcbf7d'), ObjectId('69e23677c033aea568bcbf7e'), ObjectId('69e23677c033aea568bcbf7f'), ObjectId('69e23677c033aea568bcbf80'), ObjectId('69e23677c033aea568bcbf81'), ObjectId('69e23677c033aea568bcbf82'), ObjectId('69e23677c033aea568bcbf83'), ObjectId('69e23677c033aea568bcbf84'), ObjectId('69e23677c033aea568bcbf85'), ObjectId('69e23677c033aea568bcbf86'), ObjectId('69e23677c033aea568bcbf87'), ObjectId('69e23677c033aea568bcbf88'), ObjectId('69e23677c033aea568bcbf89'), ObjectId('69e23677c033aea568bcbf8a'), ObjectId('69e23677c033aea568bcbf8b'), ObjectId('69e23677c033aea568bcbf8c'), ObjectId('69e23677c033aea568bcbf8d'), ObjectId('69e23677c033aea568bcbf8e'), ObjectId('69e23677c033aea568bcbf8f'), ObjectId('69e23677c033aea568bcbf90'), ObjectId('69e23677c033aea568bcbf91'), ObjectId('69e23677c033aea568bcbf92'), ObjectId('69e23677c033aea568bcbf93'), ObjectId('69e23677c033aea568bcbf

In [9]:
# Query with search index

from pymongo.operations import SearchIndexModel
import time

# Create your index model, then create the search index
index_name = "vector_index"
search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "numDimensions": 384,
                "path": "embedding",
                "similarity": "cosine"
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)



collection.create_search_index(model=search_index_model)
print("Vector search index created")

Vector search index created


In [10]:
# wait for inital sync for to complete 

print("Polling to check if the index is ready. This may take up to a minute")

predicate = None

if predicate is None:
    predicate = lambda index: index.get("queryable") is True
    

while True:
    indices = list(collection.list_search_indexes(index_name))
    if len(indices) and predicate(indices[0]):
        break
    time.sleep(5)

print(index_name + "is ready for querying")
    

Polling to check if the index is ready. This may take up to a minute
vector_indexis ready for querying


In [11]:
query_embedding = get_embedding("AI Technology")
query_embedding

[-0.056434500962495804,
 0.0043808454647660255,
 0.0005034097703173757,
 -0.03747867792844772,
 -0.014615318737924099,
 -0.013229303061962128,
 0.08044428378343582,
 0.06846938282251358,
 0.022890156134963036,
 -0.01440771110355854,
 -0.02283361740410328,
 0.024914013221859932,
 0.048138342797756195,
 0.003775110002607107,
 -0.05183621868491173,
 0.011305843479931355,
 -0.032184015959501266,
 -0.02369539998471737,
 -0.09368978440761566,
 -0.12984995543956757,
 0.04179004579782486,
 -0.007480152882635593,
 0.013280052691698074,
 -0.06713928282260895,
 -0.03448047861456871,
 0.07540684193372726,
 0.009522365406155586,
 -0.11432398855686188,
 0.0041964370757341385,
 -0.035460464656353,
 0.02577194757759571,
 0.012851427309215069,
 0.04413685202598572,
 -0.004286503419280052,
 -0.12459556758403778,
 0.02621043473482132,
 -0.051813654601573944,
 0.004533957690000534,
 0.07225211709737778,
 -0.017715854570269585,
 -0.049812201410532,
 -0.07045702636241913,
 0.03457007184624672,
 -0.078498229

In [12]:
results = collection.aggregate([
    {
        "$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 384,
            "limit": 5
        }
    }
])

In [13]:
# Define a function to run vector search query

def get_query_results(query):
    query_embedding = get_embedding(query, input_type="query")

    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 384,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "text": 1
            }
        }
    ]

    results = collection.aggregate(pipeline)
    return list(results)

In [14]:
res = get_query_results("mongoDB vector search")

print("\n")

print(res)



[{'text': 'of MongoDB 8.0—with significant performance improvements such as faster reads and updates, along with significantly\nfaster bulk inserts and time series queries—and the general availability of Atlas Stream Processing to build sophisticated,\nevent-driven applications with real-time data.'}, {'text': 'of MongoDB 8.0—with significant performance improvements such as faster reads and updates, along with significantly\nfaster bulk inserts and time series queries—and the general availability of Atlas Stream Processing to build sophisticated,\nevent-driven applications with real-time data.'}, {'text': 'of MongoDB 8.0—with significant performance improvements such as faster reads and updates, along with significantly\nfaster bulk inserts and time series queries—and the general availability of Atlas Stream Processing to build sophisticated,\nevent-driven applications with real-time data.'}, {'text': 'of MongoDB 8.0—with significant performance improvements such as faster reads and

In [15]:
query = "What are mongodb latest announcements ?"
context_doc = get_query_results(query)
context_string = " ".join([doc['text'] for doc in context_doc])

# Construct prompt for the llm using the retrieved documents as the context

prompt = f"""

use the following pieces of context to answer the question at the end.
{context_string}

Question: {query}

"""

response = llm.invoke([prompt])
print(response.content)

**MongoDB’s most recent public announcement (May 30 2024)**

- **Quarter‑end financial results** – MongoDB released its first‑quarter 2024 earnings for the period ending April 30, 2024.  
- **Key highlights** –  
  - **Atlas revenue grew 32 % year‑over‑year**.  
  - The company noted a **slower‑than‑expected start** to the quarter (the press release cuts off before the full explanation).  
- **Release details** – The results were issued via a PRNewswire press release on May 30 2024 and are available on MongoDB’s investor relations website.  

In short, MongoDB’s latest announcement is the Q1 2024 earnings report, underscoring strong Atlas growth and a brief slowdown in the quarter’s early performance.
